<a href="https://colab.research.google.com/github/jacksonsimon101910-tech/quant-backtesting-engine/blob/main/Backtester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings

warnings.filterwarnings("ignore")   # keep output tidy


# NOTE: Configuration parameters are moved to the backtesting function arguments.
# The constants defined here will be used as defaults or for the initial run.

# ── 1. CONFIGURATION ──────────────────────────────────────────────────────────
# All tunable parameters live here — change these to experiment.

TICKERS         = ["SPY", "QQQ", "IWM", "GLD", "TLT"] # Assets to backtest
START_DATE      = "2015-01-01"  # Backtest start
END_DATE        = "2024-12-31"  # Backtest end
FAST_SMA        = 50            # Short-term moving average (days)
SLOW_SMA        = 200           # Long-term moving average (days)
INITIAL_CAPITAL = 10_000.0      # Starting portfolio cash (USD)
COMMISSION_PCT  = 0.001         # 0.10 % per trade (each buy AND sell)
RISK_FREE_RATE  = 0.04          # Annual risk-free rate for Sharpe ratio
SLIPPAGE_PCT    = 0.0005        # 0.05 % slippage per trade (each buy AND sell)

# Position Sizing and Risk Management
POSITION_SIZE_PCT        = 0.01  # % of capital to risk per trade (for static sizing)
MAX_EXPOSURE_PCT         = 0.10  # Max % of total portfolio value in a single position
VOLATILITY_ADJUSTED_SIZING = True # Enable/disable volatility-adjusted sizing
ATR_PERIOD               = 14    # Period for Average True Range calculation
ATR_MULTIPLIER           = 2.0   # Multiplier for ATR to determine stop distance/position size

# Stop-Loss and Take-Profit
STOP_LOSS_PCT   = 0.05  # Percentage below entry price for stop-loss
TAKE_PROFIT_PCT = 0.15  # Percentage above entry price for take-profit

def run_sma_backtest(ticker, start_date, end_date, fast_sma, slow_sma,
                     initial_capital, commission_pct, risk_free_rate, slippage_pct,
                     position_size_pct, max_exposure_pct, volatility_adjusted_sizing,
                     atr_period, atr_multiplier, stop_loss_pct, take_profit_pct,
                     log_output=True, plot_output=False, plot_title_suffix=""):
    """
    Runs an SMA crossover backtest and returns performance metrics and portfolio data.
    """
    if log_output:
        print(f"\n{'='*60}")
        print(f"  Backtesting {ticker}  |  {start_date} -> {end_date}")
        print(f"  Strategy : SMA({fast_sma}) x SMA({slow_sma})")
        print(f"{'='*60}\n")

    def _synthetic_spy(start, end, seed=42):
        """
        Generate a realistic synthetic SPY price series.
        """
        if log_output:
            print("Note: yfinance not available — using synthetic SPY data for demo.")
            print("      Install it with:  pip install yfinance\n")
        rng         = np.random.default_rng(seed)
        dates       = pd.bdate_range(start=start, end=end)   # business days only
        n           = len(dates)
        mu_daily    = 0.10  / 252          # ~10 % annual drift
        sigma_daily = 0.18  / np.sqrt(252) # ~18 % annual volatility
        log_returns = rng.normal(mu_daily - 0.5 * sigma_daily**2, sigma_daily, n)
        prices      = 200.0 * np.exp(np.cumsum(log_returns))  # start near $200
        return pd.DataFrame({"Close": prices}, index=dates)

    try:
        import yfinance as yf
        if log_output:
            print("Downloading price data ...")
        raw = yf.download(ticker, start=start_date, end=end_date,
                          auto_adjust=True, progress=False)
        if raw.empty:
            raise ValueError("Empty download")
        df = raw[["Close", "High", "Low"]].copy() # Need High/Low for ATR
        df.columns = ["Close", "High", "Low"]
        df.dropna(inplace=True)
        data_source = "Yahoo Finance"
    except Exception:
        df = _synthetic_spy(start_date, end_date) # Synthetic does not have High/Low
        df["High"] = df["Close"] * 1.005 # Simple High/Low for synthetic
        df["Low"]  = df["Close"] * 0.995
        data_source = "Synthetic (GBM)"

    if log_output:
        print(f"Loaded {len(df)} trading days  [{df.index[0].date()} to {df.index[-1].date()}]")
        print(f"Data source: {data_source}\n")


    # ── 3. COMPUTE MOVING AVERAGES & ATR ──────────────────────────────────────────
    df["SMA_fast"] = df["Close"].rolling(window=fast_sma).mean()
    df["SMA_slow"] = df["Close"].rolling(window=slow_sma).mean()

    # Calculate ATR for volatility-adjusted sizing and stop-loss
    if volatility_adjusted_sizing or stop_loss_pct > 0:
        # True Range calculation
        df['H-L']   = df['High'] - df['Low']
        df['H-PC']  = abs(df['High'] - df['Close'].shift(1))
        df['L-PC']  = abs(df['Low'] - df['Close'].shift(1))
        df['TR']    = df[['H-L', 'H-PC', 'L-PC']].max(axis=1)
        df['ATR']   = df['TR'].rolling(window=atr_period).mean()

    df.dropna(inplace=True)


    # ── 4. GENERATE BUY / SELL SIGNALS (Lookahead Bias Removed) ───────────────────
    # 'position' at day 't' is what we *would* do if we could trade on day 't's close
    df["position"] = (df["SMA_fast"] > df["SMA_slow"]).astype(int)

    # To eliminate lookahead bias, we need to make trade decisions based on
    # the signal from the *previous* day (t-1) and execute on the *current* day (t).
    # 'position_for_trade_decision' at day 't' reflects the position decided at day 't-1' close.
    df['position_for_trade_decision'] = df['position'].shift(1)

    # 'trade_signal' at day 't' is the signal generated from day 't-1' to be acted upon on day 't'.
    # 1: Buy signal (position changed from 0 to 1 based on t-1's close)
    # -1: Sell signal (position changed from 1 to 0 based on t-1's close)
    # 0: No change
    df['trade_signal'] = df['position_for_trade_decision'].diff()

    # Drop any rows with NaN values introduced by shifting (first few rows)
    df.dropna(inplace=True)


    # ── 5. SIMULATE TRADES ───────────────────────────────────────────────────────
    cash        = initial_capital
    shares      = 0.0
    portfolio   = []
    trade_log   = []
    active_trade = {"entry_date": None, "entry_price": None, "shares": 0,
                    "stop_loss": None, "take_profit": None}

    for date, row in df.iterrows():
        price  = float(row["Close"])
        high   = float(row["High"]) if "High" in row.index else price * 1.005 # Fallback for synthetic
        low    = float(row["Low"]) if "Low" in row.index else price * 0.995 # Fallback for synthetic
        signal = row["trade_signal"] # Use the lookahead-free signal for decision
        current_atr = row['ATR'] if 'ATR' in row.index and pd.notna(row['ATR']) else None

        # Check for stop-loss or take-profit triggers for existing position
        if active_trade["shares"] > 0:
            # Stop-Loss Check
            if active_trade["stop_loss"] is not None and low <= active_trade["stop_loss"]:
                exec_price = active_trade["stop_loss"] * (1 - slippage_pct) # Execute at stop-loss price (worsened)
                gross       = active_trade["shares"] * exec_price
                commission  = gross * commission_pct
                cash        += gross - commission
                pnl         = (exec_price - active_trade["entry_price"]) * active_trade["shares"] - commission
                holding_period = (date - active_trade["entry_date"]).days
                trade_log.append({
                    "date": date, "type": "SELL (SL)", "price": exec_price,
                    "shares": active_trade["shares"], "commission": commission,
                    "pnl": pnl, "entry": active_trade["entry_price"], "holding_period": holding_period
                })
                active_trade = {"entry_date": None, "entry_price": None, "shares": 0,
                                "stop_loss": None, "take_profit": None}
                signal = 0 # Suppress other signals on this day if SL hit

            # Take-Profit Check (only if stop-loss not hit)
            elif active_trade["take_profit"] is not None and high >= active_trade["take_profit"]:
                exec_price = active_trade["take_profit"] * (1 - slippage_pct) # Execute at take-profit price (worsened)
                gross       = active_trade["shares"] * exec_price
                commission  = gross * commission_pct
                cash        += gross - commission
                pnl         = (exec_price - active_trade["entry_price"]) * active_trade["shares"] - commission
                holding_period = (date - active_trade["entry_date"]).days
                trade_log.append({
                    "date": date, "type": "SELL (TP)", "price": exec_price,
                    "shares": active_trade["shares"], "commission": commission,
                    "pnl": pnl, "entry": active_trade["entry_price"], "holding_period": holding_period
                })
                active_trade = {"entry_date": None, "entry_price": None, "shares": 0,
                                "stop_loss": None, "take_profit": None}
                signal = 0 # Suppress other signals on this day if TP hit

        # Process new buy/sell signals
        if signal == 1 and active_trade["shares"] == 0 and cash > 0: # Buy signal and not already in a position
            exec_price  = price * (1 + slippage_pct)

            # Calculate target position size
            if volatility_adjusted_sizing and current_atr is not None:
                risk_per_share = current_atr * atr_multiplier
                if risk_per_share > 0:
                    # Risk a percentage of capital, determine shares based on ATR
                    shares_to_buy_from_risk = (cash * position_size_pct) / risk_per_share
                else:
                    shares_to_buy_from_risk = 0

                # Also consider max exposure percentage
                max_shares_from_exposure = (cash * max_exposure_pct) / exec_price
                shares_to_buy = min(shares_to_buy_from_risk, max_shares_from_exposure)
            else: # Static position sizing
                # Determine shares based on max exposure percent of capital
                shares_to_buy = (cash * max_exposure_pct) / exec_price

            # Ensure we don't buy more shares than available capital allows after commission
            potential_cost = shares_to_buy * exec_price
            commission_on_buy = potential_cost * commission_pct
            if potential_cost + commission_on_buy > cash:
                # Adjust shares to fit within available cash
                shares_to_buy = (cash / (exec_price * (1 + commission_pct)))

            # Execute buy if shares_to_buy is significant
            if shares_to_buy > 0.0001: # Avoid tiny fractional shares
                cost        = shares_to_buy * exec_price
                commission  = cost * commission_pct
                cash        -= (cost + commission)

                # Set active trade details
                active_trade["entry_date"]  = date
                active_trade["entry_price"] = exec_price
                active_trade["shares"]      = shares_to_buy

                # Calculate Stop-Loss and Take-Profit levels
                if stop_loss_pct > 0:
                    active_trade["stop_loss"] = exec_price * (1 - stop_loss_pct)
                if take_profit_pct > 0:
                    active_trade["take_profit"] = exec_price * (1 + take_profit_pct)

                trade_log.append({
                    "date": date, "type": "BUY", "price": exec_price,
                    "shares": shares_to_buy, "commission": commission, "entry_date": date,
                    "holding_period": np.nan, # Included for consistency
                    "stop_loss": active_trade["stop_loss"],
                    "take_profit": active_trade["take_profit"]
                })

        elif signal == -1 and active_trade["shares"] > 0: # Sell signal and in a position
            exec_price  = price * (1 - slippage_pct)
            gross       = active_trade["shares"] * exec_price
            commission  = gross * commission_pct
            cash        += gross - commission
            pnl         = (exec_price - active_trade["entry_price"]) * active_trade["shares"] - commission
            holding_period = (date - active_trade["entry_date"]).days
            trade_log.append({
                "date": date, "type": "SELL", "price": exec_price,
                "shares": active_trade["shares"], "commission": commission,
                "pnl": pnl, "entry": active_trade["entry_price"], "holding_period": holding_period
            })
            active_trade = {"entry_date": None, "entry_price": None, "shares": 0,
                            "stop_loss": None, "take_profit": None}

        # Current portfolio value is cash plus value of shares held at current price.
        portfolio.append({"date": date, "value": cash + active_trade["shares"] * price})

    # If there's an open position at the end of the backtest, close it out
    if active_trade["shares"] > 0:
        last_date = df.index[-1]
        last_price = df["Close"].iloc[-1]
        exec_price = last_price * (1 - slippage_pct)
        gross       = active_trade["shares"] * exec_price
        commission  = gross * commission_pct
        cash        += gross - commission
        pnl         = (exec_price - active_trade["entry_price"]) * active_trade["shares"] - commission
        holding_period = (last_date - active_trade["entry_date"]).days
        trade_log.append({
            "date": last_date, "type": "SELL (EOD)", "price": exec_price,
            "shares": active_trade["shares"], "commission": commission,
            "pnl": pnl, "entry": active_trade["entry_price"], "holding_period": holding_period
        })

    portfolio_df       = pd.DataFrame(portfolio).set_index("date")
    portfolio_df.index = pd.to_datetime(portfolio_df.index)
    trades_df          = pd.DataFrame(trade_log) if trade_log else pd.DataFrame()
    if not trades_df.empty:
        trades_df      = trades_df.set_index("date")


    # ── 6. CALCULATE PERFORMANCE METRICS ─────────────────────────────────────────
    final_value  = portfolio_df["value"].iloc[-1]
    total_return = (final_value - initial_capital) / initial_capital * 100

    bh_shares = (initial_capital * (1 - commission_pct)) / float(df["Close"].iloc[0])
    bh_final  = bh_shares * float(df["Close"].iloc[-1])
    bh_return = (bh_final - initial_capital) / initial_capital * 100
    bh_series = bh_shares * df["Close"].reindex(portfolio_df.index)

    daily_ret  = portfolio_df["value"].pct_change().dropna()
    excess_ret = daily_ret - risk_free_rate / 252
    sharpe     = (excess_ret.mean() / excess_ret.std()) * np.sqrt(252)

    rolling_max = portfolio_df["value"].cummax()
    drawdown    = (portfolio_df["value"] - rolling_max) / rolling_max * 100
    max_dd      = drawdown.min()

    sell_trades = (trades_df[trades_df["type"].str.contains("SELL")]
                   if not trades_df.empty else pd.DataFrame())
    if not sell_trades.empty:
        winning  = (sell_trades["pnl"] > 0).sum()
        win_rate = winning / len(sell_trades) * 100
        n_trades = len(sell_trades)
    else:
        win_rate = 0.0
        n_trades = 0

    # Calculate advanced metrics
    advanced_metrics = calculate_advanced_metrics(portfolio_df, trades_df, initial_capital, risk_free_rate, n_trades)

    if log_output:
        print("=" * 44)
        print("  PERFORMANCE SUMMARY")
        print("=" * 44)
        print(f"  Initial Capital    : ${initial_capital:>12,.2f}")
        print(f"  Final Value        : ${final_value:>12,.2f}")
        print(f"  Strategy Return    : {total_return:>+11.2f} %")
        print(f"  CAGR               : {advanced_metrics['cagr']:>+11.2f} %")
        print(f"  Buy-and-Hold Ret.  : {bh_return:>+11.2f} %")
        print(f"  Volatility         : {advanced_metrics['volatility']:>11.2f} %")
        print(f"  Sharpe Ratio       : {sharpe:>12.3f}")
        print(f"  Sortino Ratio      : {advanced_metrics['sortino_ratio']:>12.3f}")
        print(f"  Max Drawdown       : {max_dd:>11.2f} %")
        print(f"  Calmar Ratio       : {advanced_metrics['calmar_ratio']:>12.3f}")
        print(f"  Win Rate           : {win_rate:>11.1f} %  ({n_trades} trades)")
        print(f"  Profit Factor      : {advanced_metrics['profit_factor']:>12.2f}")
        print(f"  Avg Trade Return   : ${advanced_metrics['avg_trade_return']:>11.2f}")
        print(f"  Expectancy         : ${advanced_metrics['expectancy']:>11.2f}")
        print(f"  Avg Holding Period : {advanced_metrics['avg_holding_period']:>11.1f} days")
        print("=" * 44)

        if not sell_trades.empty:
            print("\n  TRADE LOG (last 10 closed trades):")
            print(f"  {'Date':<12} {'Type':<11} {'Price':>8}  {'Shares':>8}  {'P&L':>10}  {'Hold':>5}") # Increased Type column width
            print("  " + "-" * 64) # Adjusted separator length
            for date, t in trades_df.tail(10).iterrows():
                pnl_str = f"${t['pnl']:>+9.2f}" if "pnl" in t and pd.notna(t['pnl']) else " " * 11
                hold_str = f"{t['holding_period']:>5.0f}" if "holding_period" in t and pd.notna(t['holding_period']) else " " * 5
                print(f"  {str(date.date()):<12} {t['type']:<11} {t['price']:>8.2f}"
                      f"  {t['shares']:>8.4f}  {pnl_str}  {hold_str}")
        print()

    if plot_output:
        # Colour palette
        BG       = "#181c26"
        C_PRICE  = "#dce1e7"
        C_FAST   = "#f6c90e"   # gold  (fast SMA)
        C_SLOW   = "#3fa7d6"   # steel blue (slow SMA)
        C_BUY    = "#4caf50"   # green buy triangles
        C_SELL   = "#f44336"   # red sell triangles
        C_EQUITY = "#9c6ade"   # purple equity curve
        C_BH     = "#546e8a"   # muted blue benchmark
        C_DD     = "#f44336"   # red drawdown

        fig = plt.figure(figsize=(14, 11))
        fig.patch.set_facecolor("#0f1117")
        gs = gridspec.GridSpec(3, 1, figure=fig, height_ratios=[3, 2, 1.2], hspace=0.08)

        AX_PRICE  = fig.add_subplot(gs[0])
        AX_EQUITY = fig.add_subplot(gs[1], sharex=AX_PRICE)
        AX_DD      = fig.add_subplot(gs[2], sharex=AX_PRICE)

        for ax in [AX_PRICE, AX_EQUITY, AX_DD]:
            ax.set_facecolor(BG)
            ax.tick_params(colors="#aaaaaa", labelsize=9)
            ax.yaxis.label.set_color("#aaaaaa")
            for spine in ax.spines.values():
                spine.set_edgecolor("#333344")
            ax.grid(color="#262636", linewidth=0.5, linestyle="--")

        # Panel 1: Price + moving averages + trade signals
        AX_PRICE.plot(df.index, df["Close"],    color=C_PRICE, lw=1.0, label=f"{ticker} Close")
        AX_PRICE.plot(df.index, df["SMA_fast"], color=C_FAST,  lw=1.5, label=f"SMA({fast_sma})")
        AX_PRICE.plot(df.index, df["SMA_slow"], color=C_SLOW,  lw=1.5, label=f"SMA({slow_sma})")

        # Plot original signals for visualization (not for trade execution)
        buys_original  = df[df["position"].diff() ==  1]
        sells_original = df[df["position"].diff() == -1]
        AX_PRICE.scatter(buys_original.index,  buys_original["Close"],  marker="^", color=C_BUY,  s=80,
                         zorder=5, label="Buy (original signal)")
        AX_PRICE.scatter(sells_original.index, sells_original["Close"], marker="v", color=C_SELL, s=80,
                         zorder=5, label="Sell (original signal)")

        AX_PRICE.set_ylabel("Price (USD)", fontsize=10)
        title_full = (
            f"{ticker}  |  SMA({fast_sma}) x SMA({slow_sma}) Crossover  "
            f"[{start_date} to {end_date}]{plot_title_suffix}"
        )
        AX_PRICE.set_title(
            title_full,
            color="white", fontsize=12, pad=10, fontweight="bold"
        )
        AX_PRICE.legend(loc="upper left", fontsize=8, framealpha=0.25,
                        facecolor="#0f1117", labelcolor="white")

        # Panel 2: Equity curve vs. buy-and-hold benchmark
        AX_EQUITY.plot(portfolio_df.index, portfolio_df["value"],
                       color=C_EQUITY, lw=1.6, label=f"Strategy   {total_return:+.1f}%")
        AX_EQUITY.plot(bh_series.index, bh_series.values,
                       color=C_BH, lw=1.4, linestyle="--",
                       label=f"Buy & Hold  {bh_return:+.1f}%")
        AX_EQUITY.axhline(initial_capital, color="#555566", lw=0.8, linestyle=":")
        AX_EQUITY.set_ylabel("Portfolio Value ($", fontsize=10)
        AX_EQUITY.legend(loc="upper left", fontsize=8, framealpha=0.25,
                         facecolor="#0f1117", labelcolor="white")

        stats_label = (f"Sharpe: {sharpe:.2f}   |   Max Drawdown: {max_dd:.1f}%   |   Win Rate: {win_rate:.0f}%  "
                       f"({n_trades} trades)")
        AX_EQUITY.text(0.99, 0.04, stats_label, transform=AX_EQUITY.transAxes, fontsize=8, color="#bbbbbb", ha="right", bbox=dict(boxstyle="round,pad=0.35", fc="#0f1117", alpha=0.55))

        # Panel 3: Drawdown waterfall
        AX_DD.fill_between(portfolio_df.index, drawdown, 0, color=C_DD, alpha=0.35)
        AX_DD.plot(portfolio_df.index, drawdown, color=C_DD, lw=0.9)
        AX_DD.axhline(max_dd, color=C_DD, lw=0.8, linestyle="--", alpha=0.7, label=f"Max DD: {max_dd:.1f}%")
        AX_DD.set_ylabel("Drawdown (%)", fontsize=10)
        AX_DD.set_xlabel("Date", fontsize=10, color="#aaaaaa")
        AX_DD.legend(loc="lower left", fontsize=8, framealpha=0.25, facecolor="#0f1117", labelcolor="white")

        plt.setp(AX_PRICE.get_xticklabels(),  visible=False)
        plt.setp(AX_EQUITY.get_xticklabels(), visible=False)

        output_path = "backtest_results.png"
        plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
        plt.show()
        if log_output:
            print(f"Chart saved -> {output_path}\n")

    return {
        "total_return": total_return,
        "bh_return": bh_return,
        "sharpe_ratio": sharpe,
        "max_drawdown": max_dd,
        "win_rate": win_rate,
        "n_trades": n_trades,
        "portfolio_df": portfolio_df,
        "bh_series": bh_series,
        **advanced_metrics # Include all advanced metrics
    }

In [ ]:
# @title
def calculate_advanced_metrics(portfolio_df, trades_df, initial_capital, risk_free_rate, n_trades):
    metrics = {}

    # Total Return (already calculated in backtest functions, but useful here for CAGR)
    final_value  = portfolio_df["value"].iloc[-1]
    total_return = (final_value - initial_capital) / initial_capital

    # Annualized Return (CAGR)
    # Calculate number of years based on the portfolio DataFrame index
    if len(portfolio_df) > 1:
        first_date = portfolio_df.index.min()
        last_date = portfolio_df.index.max()
        num_years = (last_date - first_date).days / 365.25
        if num_years > 0:
            cagr = ((final_value / initial_capital) ** (1 / num_years)) - 1
        else:
            cagr = 0.0 # Handle cases with less than a year of data
    else:
        cagr = 0.0
    metrics['cagr'] = cagr * 100 # as a percentage

    # Volatility (Annualized Standard Deviation of Daily Returns)
    daily_ret = portfolio_df["value"].pct_change().dropna()
    volatility = daily_ret.std() * np.sqrt(252) # 252 trading days in a year
    metrics['volatility'] = volatility * 100 # as a percentage

    # Sortino Ratio
    # Only consider downside deviation (negative returns)
    downside_returns = daily_ret[daily_ret < 0]
    if len(downside_returns) > 0:
        downside_std = downside_returns.std()
        if downside_std != 0:
            excess_ret = daily_ret - risk_free_rate / 252
            sortino_ratio = (excess_ret.mean() / downside_std) * np.sqrt(252)
        else:
            sortino_ratio = np.nan # No downside deviation, handle division by zero
    else:
        sortino_ratio = np.nan # No negative returns
    metrics['sortino_ratio'] = sortino_ratio

    # Calmar Ratio
    # Max Drawdown should be passed as a positive value for this calculation
    rolling_max = portfolio_df["value"].cummax()
    drawdown     = (portfolio_df["value"] - rolling_max) / rolling_max
    max_dd       = drawdown.min()
    if max_dd < 0:
        metrics['calmar_ratio'] = cagr / abs(max_dd) if cagr != 0 else 0.0
    else:
        metrics['calmar_ratio'] = np.nan # No drawdown, or positive drawdown means infinite Calmar Ratio

    # Profit Factor
    # Ensure trades_df is not empty and 'type' column exists before filtering
    if not trades_df.empty and 'type' in trades_df.columns:
        sell_trades = trades_df[trades_df["type"].str.contains("SELL", na=False)]
        if not sell_trades.empty:
            winning_trades_pnl = sell_trades[sell_trades["pnl"] > 0]["pnl"].sum()
            losing_trades_pnl = abs(sell_trades[sell_trades["pnl"] < 0]["pnl"].sum())
            metrics['profit_factor'] = winning_trades_pnl / losing_trades_pnl if losing_trades_pnl > 0 else np.inf

            # Average Trade Return
            metrics['avg_trade_return'] = sell_trades["pnl"].mean()

            # Expectancy (Avg win * Win rate) - (Avg loss * Loss rate)
            winning_pnl = sell_trades[sell_trades['pnl'] > 0]['pnl']
            losing_pnl = sell_trades[sell_trades['pnl'] < 0]['pnl']

            avg_win = winning_pnl.mean() if not winning_pnl.empty else 0
            avg_loss = losing_pnl.mean() if not losing_pnl.empty else 0
            win_rate = len(winning_pnl) / len(sell_trades) if len(sell_trades) > 0 else 0
            loss_rate = 1 - win_rate
            metrics['expectancy'] = (avg_win * win_rate) + (avg_loss * loss_rate)

            # Average Holding Period (only for closed trades)
            # Ensure 'holding_period' column exists and is numeric
            if 'holding_period' in sell_trades.columns:
                metrics['avg_holding_period'] = sell_trades['holding_period'].mean()
            else:
                metrics['avg_holding_period'] = np.nan
        else:
            metrics['profit_factor'] = np.nan
            metrics['avg_trade_return'] = np.nan
            metrics['expectancy'] = np.nan
            metrics['avg_holding_period'] = np.nan
    else:
        metrics['profit_factor'] = np.nan
        metrics['avg_trade_return'] = np.nan
        metrics['expectancy'] = np.nan
        metrics['avg_holding_period'] = np.nan

    # Convert CAGR, Volatility to non-percentage for Calmar/Sharpe-like calculations if needed externally
    # But for display, keeping them as percentage is fine.

    return metrics

### Re-running Walk-Forward Optimization for SMA Strategy

After fixing the `KeyError`, let's re-run the walk-forward optimization for the SMA strategy.

In [ ]:
# Re-run SMA walk-forward optimization
print(f"Running walk-forward optimization for SMA on {WF_TICKER}...")
sma_walk_forward_results = run_walk_forward_optimization(
    ticker=WF_TICKER,
    strategy_type='SMA',
    start_date=WF_START_DATE,
    end_date=WF_END_DATE,
    in_sample_period_years=WF_IN_SAMPLE_YEARS,
    out_of_sample_period_years=WF_OUT_OF_SAMPLE_YEARS,
    step_years=WF_STEP_YEARS,
    optimization_params_range=WF_SMA_OPTIMIZATION_PARAMS,
    initial_capital=INITIAL_CAPITAL,
    commission_pct=COMMISSION_PCT,
    risk_free_rate=RISK_FREE_RATE,
    slippage_pct=SLIPPAGE_PCT,
    position_size_pct=POSITION_SIZE_PCT,
    max_exposure_pct=MAX_EXPOSURE_PCT,
    volatility_adjusted_sizing=VOLATILITY_ADJUSTED_SIZING,
    atr_period=ATR_PERIOD,
    atr_multiplier=ATR_MULTIPLIER,
    stop_loss_pct=STOP_LOSS_PCT,
    take_profit_pct=TAKE_PROFIT_PCT
)

print("\nSMA Walk-Forward Results:")
display(sma_walk_forward_results.head())

Running walk-forward optimization for SMA on SPY...

Starting Walk-Forward Optimization for SPY with SMA strategy...

  WALK-FORWARD WINDOW 1
  In-Sample: 2005-01-01 to 2009-12-31
  Out-of-Sample: 2010-01-01 to 2010-12-31
  Performing in-sample optimization...
  Best In-Sample Params: {'fast_sma': 20, 'slow_sma': 175} (Sharpe: -3.852)
  Running out-of-sample validation...
  Out-of-Sample Sharpe: -0.525

  WALK-FORWARD WINDOW 2
  In-Sample: 2006-01-01 to 2010-12-31
  Out-of-Sample: 2011-01-01 to 2011-12-31
  Performing in-sample optimization...
  Best In-Sample Params: {'fast_sma': 20, 'slow_sma': 100} (Sharpe: -3.633)
  Running out-of-sample validation...
  Out-of-Sample Sharpe: -6.727

  WALK-FORWARD WINDOW 3
  In-Sample: 2007-01-01 to 2011-12-31
  Out-of-Sample: 2012-01-01 to 2012-12-31
  Performing in-sample optimization...
  Best In-Sample Params: {'fast_sma': 30, 'slow_sma': 125} (Sharpe: -4.257)
  Running out-of-sample validation...
  Out-of-Sample Sharpe: -3.644

  WALK-FORWARD 

,Window,Strategy_Type,Ticker,In_Sample_Start,In_Sample_End,OOS_Start,OOS_End,Best_IS_Params,IS_Sharpe,OOS_Sharpe,OOS_Return,OOS_Max_Drawdown
0,1,SMA,SPY,2005-01-01,2009-12-31,2010-01-01,2010-12-31,"{'fast_sma': 20, 'slow_sma': 175}",-3.851538,-5.254106e-01,1.005789,-0.394753
1,2,SMA,SPY,2006-01-01,2010-12-31,2011-01-01,2011-12-31,"{'fast_sma': 20, 'slow_sma': 100}",-3.632638,-6.726792e+00,-1.045742,-1.174634
2,3,SMA,SPY,2007-01-01,2011-12-31,2012-01-01,2012-12-31,"{'fast_sma': 30, 'slow_sma': 125}",-4.256528,-3.644349e+00,0.139346,-0.770170
3,4,SMA,SPY,2008-01-01,2012-12-31,2013-01-01,2013-12-31,"{'fast_sma': 40, 'slow_sma': 200}",-3.417560,-1.533489e+16,0.000000,0.000000
4,5,SMA,SPY,2009-01-01,2013-12-31,2014-01-01,2014-12-31,"{'fast_sma': 60, 'slow_sma': 125}",-3.834987,-1.543121e+16,0.000000,0.000000


### Re-running Walk-Forward Optimization for RSI Strategy

And now, let's re-run the walk-forward optimization for the RSI strategy.

In [ ]:
# Re-run RSI walk-forward optimization
print(f"Running walk-forward optimization for RSI on {WF_TICKER}...")
rsi_walk_forward_results = run_walk_forward_optimization(
    ticker=WF_TICKER,
    strategy_type='RSI',
    start_date=WF_START_DATE,
    end_date=WF_END_DATE,
    in_sample_period_years=WF_IN_SAMPLE_YEARS,
    out_of_sample_period_years=WF_OUT_OF_SAMPLE_YEARS,
    step_years=WF_STEP_YEARS,
    optimization_params_range=WF_RSI_OPTIMIZATION_PARAMS,
    initial_capital=INITIAL_CAPITAL,
    commission_pct=COMMISSION_PCT,
    risk_free_rate=RISK_FREE_RATE,
    slippage_pct=SLIPPAGE_PCT,
    position_size_pct=POSITION_SIZE_PCT,
    max_exposure_pct=MAX_EXPOSURE_PCT,
    volatility_adjusted_sizing=VOLATILITY_ADJUSTED_SIZING,
    atr_period=ATR_PERIOD,
    atr_multiplier=ATR_MULTIPLIER,
    stop_loss_pct=STOP_LOSS_PCT,
    take_profit_pct=TAKE_PROFIT_PCT
)

print("\nRSI Walk-Forward Results:")
display(rsi_walk_forward_results.head())

Running walk-forward optimization for RSI on SPY...

Starting Walk-Forward Optimization for SPY with RSI strategy...

  WALK-FORWARD WINDOW 1
  In-Sample: 2005-01-01 to 2009-12-31
  Out-of-Sample: 2010-01-01 to 2010-12-31
  Performing in-sample optimization...
  Best In-Sample Params: {'rsi_period': 10, 'rsi_buy_threshold': 25, 'rsi_sell_threshold': 65} (Sharpe: -6.011)
  Running out-of-sample validation...
  Out-of-Sample Sharpe: -3.085

  WALK-FORWARD WINDOW 2
  In-Sample: 2006-01-01 to 2010-12-31
  Out-of-Sample: 2011-01-01 to 2011-12-31
  Performing in-sample optimization...
  Best In-Sample Params: {'rsi_period': 20, 'rsi_buy_threshold': 30, 'rsi_sell_threshold': 65} (Sharpe: -5.440)
  Running out-of-sample validation...
  Out-of-Sample Sharpe: -6.111

  WALK-FORWARD WINDOW 3
  In-Sample: 2007-01-01 to 2011-12-31
  Out-of-Sample: 2012-01-01 to 2012-12-31
  Performing in-sample optimization...
  Best In-Sample Params: {'rsi_period': 18, 'rsi_buy_threshold': 35, 'rsi_sell_threshold'

,Window,Strategy_Type,Ticker,In_Sample_Start,In_Sample_End,OOS_Start,OOS_End,Best_IS_Params,IS_Sharpe,OOS_Sharpe,OOS_Return,OOS_Max_Drawdown
0,1,RSI,SPY,2005-01-01,2009-12-31,2010-01-01,2010-12-31,"{'rsi_period': 10, 'rsi_buy_threshold': 25, 'r...",-6.010532,-3.085218e+00,-0.524245,-1.872312
1,2,RSI,SPY,2006-01-01,2010-12-31,2011-01-01,2011-12-31,"{'rsi_period': 20, 'rsi_buy_threshold': 30, 'r...",-5.440354,-6.111119e+00,-0.524245,-1.016385
2,3,RSI,SPY,2007-01-01,2011-12-31,2012-01-01,2012-12-31,"{'rsi_period': 18, 'rsi_buy_threshold': 35, 'r...",-5.442826,-2.835079e+00,0.672417,-0.806182
3,4,RSI,SPY,2008-01-01,2012-12-31,2013-01-01,2013-12-31,"{'rsi_period': 18, 'rsi_buy_threshold': 25, 'r...",-8.357131,-9.276142e+15,0.000000,0.000000
4,5,RSI,SPY,2009-01-01,2013-12-31,2014-01-01,2014-12-31,"{'rsi_period': 18, 'rsi_buy_threshold': 30, 'r...",-8.313578,-4.339214e+00,1.181412,-0.534843


### Analysis of Walk-Forward Optimization Results

Below is a combined view of the walk-forward optimization results for both SMA and RSI strategies. This table summarizes the in-sample (IS) and out-of-sample (OOS) Sharpe Ratios for each rolling window. A robust strategy will typically show consistent performance between its in-sample and out-of-sample periods, or at least a positive out-of-sample Sharpe Ratio, indicating profitability on unseen data.

In [ ]:
all_wf_results = pd.concat([sma_walk_forward_results, rsi_walk_forward_results]).reset_index(drop=True)
all_wf_results['Best_IS_Params'] = all_wf_results['Best_IS_Params'].apply(lambda x: eval(x) if isinstance(x, str) else x)

# Calculate OOS - IS Sharpe Difference for robustness analysis
all_wf_results['OOS_IS_Sharpe_Diff'] = all_wf_results['OOS_Sharpe'] - all_wf_results['IS_Sharpe']

print("\nCombined Walk-Forward Optimization Results:")
display(all_wf_results.round(3))


Combined Walk-Forward Optimization Results:


,Window,Strategy_Type,Ticker,In_Sample_Start,In_Sample_End,OOS_Start,OOS_End,Best_IS_Params,IS_Sharpe,OOS_Sharpe,OOS_Return,OOS_Max_Drawdown,OOS_IS_Sharpe_Diff
0,1,SMA,SPY,2005-01-01,2009-12-31,2010-01-01,2010-12-31,"{'fast_sma': 20, 'slow_sma': 175}",-3.852,-5.250000e-01,1.006,-0.395,3.326000e+00
1,2,SMA,SPY,2006-01-01,2010-12-31,2011-01-01,2011-12-31,"{'fast_sma': 20, 'slow_sma': 100}",-3.633,-6.727000e+00,-1.046,-1.175,-3.094000e+00
2,3,SMA,SPY,2007-01-01,2011-12-31,2012-01-01,2012-12-31,"{'fast_sma': 30, 'slow_sma': 125}",-4.257,-3.644000e+00,0.139,-0.770,6.120000e-01
3,4,SMA,SPY,2008-01-01,2012-12-31,2013-01-01,2013-12-31,"{'fast_sma': 40, 'slow_sma': 200}",-3.418,-1.533489e+16,0.000,0.000,-1.533489e+16
4,5,SMA,SPY,2009-01-01,2013-12-31,2014-01-01,2014-12-31,"{'fast_sma': 60, 'slow_sma': 125}",-3.835,-1.543121e+16,0.000,0.000,-1.543121e+16
5,6,SMA,SPY,2010-01-01,2014-12-31,2015-01-01,2015-12-31,"{'fast_sma': 60, 'slow_sma': 125}",-3.835,-9.070000e+00,-0.057,-0.343,-5.235000e+00
6,7,SMA,SPY,2011-01-01,2015-12-31,2016-01-01,2016-12-31,"{'fast_sma': 40, 'slow_sma': 100}",-4.138,-1.795300e+01,-0.081,-0.129,-1.381600e+01
7,8,SMA,SPY,2012-01-01,2016-12-31,2017-01-01,2017-12-31,"{'fast_sma': 20, 'slow_sma': 125}",-4.088,-1.543121e+16,0.000,0.000,-1.543121e+16
8,9,SMA,SPY,2013-01-01,2017-12-31,2018-01-01,2018-12-31,"{'fast_sma': 40, 'slow_sma': 100}",-4.670,-4.694000e+00,-0.524,-1.076,-2.400000e-02
9,10,SMA,SPY,2014-01-01,2018-12-31,2019-01-01,2019-12-31,"{'fast_sma': 40, 'slow_sma': 100}",-4.341,-6.176691e+15,0.000,0.000,-6.176691e+15


In [ ]:
def run_rsi_backtest(ticker, start_date, end_date, rsi_period, rsi_buy_threshold, rsi_sell_threshold,
                     initial_capital, commission_pct, risk_free_rate, slippage_pct,
                     position_size_pct, max_exposure_pct, volatility_adjusted_sizing,
                     atr_period, atr_multiplier, stop_loss_pct, take_profit_pct,
                     log_output=True, plot_output=False, plot_title_suffix=""):
    """
    Runs an RSI mean reversion backtest and returns performance metrics and portfolio data.
    """
    if log_output:
        print(f"\n{'='*60}")
        print(f"  Backtesting {ticker}  |  {start_date} -> {end_date}")
        print(f"  Strategy : RSI({rsi_period}) Buy < {rsi_buy_threshold} / Sell > {rsi_sell_threshold}")
        print(f"{'='*60}\n")

    def _synthetic_spy(start, end, seed=42):
        """
        Generate a realistic synthetic SPY price series.
        """
        if log_output:
            print("Note: yfinance not available — using synthetic SPY data for demo.")
            print("      Install it with:  pip install yfinance\n")
        rng         = np.random.default_rng(seed)
        dates       = pd.bdate_range(start=start, end=end)   # business days only
        n           = len(dates)
        mu_daily    = 0.10  / 252          # ~10 % annual drift
        sigma_daily = 0.18  / np.sqrt(252) # ~18 % annual volatility
        log_returns = rng.normal(mu_daily - 0.5 * sigma_daily**2, sigma_daily, n)
        prices      = 200.0 * np.exp(np.cumsum(log_returns))  # start near $200
        return pd.DataFrame({"Close": prices}, index=dates)

    try:
        import yfinance as yf
        if log_output:
            print("Downloading price data ...")
        raw = yf.download(ticker, start=start_date, end=end_date,
                          auto_adjust=True, progress=False)
        if raw.empty:
            raise ValueError("Empty download")
        df = raw[["Close", "High", "Low"]].copy() # Need High/Low for ATR
        df.columns = ["Close", "High", "Low"]
        df.dropna(inplace=True)
        data_source = "Yahoo Finance"
    except Exception:
        df = _synthetic_spy(start_date, end_date) # Synthetic does not have High/Low
        df["High"] = df["Close"] * 1.005 # Simple High/Low for synthetic
        df["Low"]  = df["Close"] * 0.995
        data_source = "Synthetic (GBM)"

    if log_output:
        print(f"Loaded {len(df)} trading days  [{df.index[0].date()} to {df.index[-1].date()}]")
        print(f"Data source: {data_source}\n")

    # 3. CALCULATE RSI & ATR
    # Calculate RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=rsi_period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=rsi_period).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # Calculate ATR for volatility-adjusted sizing and stop-loss
    if volatility_adjusted_sizing or stop_loss_pct > 0:
        df['H-L']   = df['High'] - df['Low']
        df['H-PC']  = abs(df['High'] - df['Close'].shift(1))
        df['L-PC']  = abs(df['Low'] - df['Close'].shift(1))
        df['TR']    = df[['H-L', 'H-PC', 'L-PC']].max(axis=1)
        df['ATR']   = df['TR'].rolling(window=atr_period).mean()

    df.dropna(inplace=True)

    # 4. GENERATE BUY / SELL SIGNALS (Lookahead Bias Removed)
    # 'position' at day 't' is what we *would* do if we could trade on day 't's close
    # Buy when RSI drops below rsi_buy_threshold, Sell when RSI rises above rsi_sell_threshold
    df['position'] = 0
    df.loc[df['RSI'] < rsi_buy_threshold, 'position'] = 1 # Over-sold, consider buying
    df.loc[df['RSI'] > rsi_sell_threshold, 'position'] = 0 # Over-bought, consider selling

    # Fill intermediate periods where RSI is between thresholds, maintaining current position
    # Use ffill to maintain position until a new signal
    df['position'] = df['position'].replace(to_replace=0, method='ffill').fillna(0)

    # To eliminate lookahead bias, we need to make trade decisions based on
    # the signal from the *previous* day (t-1) and execute on the *current* day (t).
    df['position_for_trade_decision'] = df['position'].shift(1)
    df['trade_signal'] = df['position_for_trade_decision'].diff()
    df.dropna(inplace=True)

    # 5. SIMULATE TRADES
    cash        = initial_capital
    shares      = 0.0
    portfolio   = []
    trade_log   = []
    active_trade = {"entry_date": None, "entry_price": None, "shares": 0,
                    "stop_loss": None, "take_profit": None}

    for date, row in df.iterrows():
        price  = float(row["Close"])
        high   = float(row["High"]) if "High" in row.index else price * 1.005 # Fallback for synthetic
        low    = float(row["Low"]) if "Low" in row.index else price * 0.995 # Fallback for synthetic
        signal = row["trade_signal"] # Use the lookahead-free signal for decision
        current_atr = row['ATR'] if 'ATR' in row.index and pd.notna(row['ATR']) else None

        # Check for stop-loss or take-profit triggers for existing position
        if active_trade["shares"] > 0:
            # Stop-Loss Check
            if active_trade["stop_loss"] is not None and low <= active_trade["stop_loss"]:
                exec_price = active_trade["stop_loss"] * (1 - slippage_pct) # Execute at stop-loss price (worsened)
                gross       = active_trade["shares"] * exec_price
                commission  = gross * commission_pct
                cash        += gross - commission
                pnl         = (exec_price - active_trade["entry_price"]) * active_trade["shares"] - commission
                holding_period = (date - active_trade["entry_date"]).days
                trade_log.append({
                    "date": date, "type": "SELL (SL)", "price": exec_price,
                    "shares": active_trade["shares"], "commission": commission,
                    "pnl": pnl, "entry": active_trade["entry_price"], "holding_period": holding_period
                })
                active_trade = {"entry_date": None, "entry_price": None, "shares": 0,
                                "stop_loss": None, "take_profit": None}
                signal = 0 # Suppress other signals on this day if SL hit

            # Take-Profit Check (only if stop-loss not hit)
            elif active_trade["take_profit"] is not None and high >= active_trade["take_profit"]:
                exec_price = active_trade["take_profit"] * (1 - slippage_pct) # Execute at take-profit price (worsened)
                gross       = active_trade["shares"] * exec_price
                commission  = gross * commission_pct
                cash        += gross - commission
                pnl         = (exec_price - active_trade["entry_price"]) * active_trade["shares"] - commission
                holding_period = (date - active_trade["entry_date"]).days
                trade_log.append({
                    "date": date, "type": "SELL (TP)", "price": exec_price,
                    "shares": active_trade["shares"], "commission": commission,
                    "pnl": pnl, "entry": active_trade["entry_price"], "holding_period": holding_period
                })
                active_trade = {"entry_date": None, "entry_price": None, "shares": 0,
                                "stop_loss": None, "take_profit": None}
                signal = 0 # Suppress other signals on this day if TP hit

        # Process new buy/sell signals
        if signal == 1 and active_trade["shares"] == 0 and cash > 0: # Buy signal and not already in a position
            exec_price  = price * (1 + slippage_pct)

            # Calculate target position size
            if volatility_adjusted_sizing and current_atr is not None:
                risk_per_share = current_atr * atr_multiplier
                if risk_per_share > 0:
                    # Risk a percentage of capital, determine shares based on ATR
                    shares_to_buy_from_risk = (cash * position_size_pct) / risk_per_share
                else:
                    shares_to_buy_from_risk = 0

                # Also consider max exposure percentage
                max_shares_from_exposure = (cash * max_exposure_pct) / exec_price
                shares_to_buy = min(shares_to_buy_from_risk, max_shares_from_exposure)
            else: # Static position sizing
                # Determine shares based on max exposure percent of capital
                shares_to_buy = (cash * max_exposure_pct) / exec_price

            # Ensure we don't buy more shares than available capital allows after commission
            potential_cost = shares_to_buy * exec_price
            commission_on_buy = potential_cost * commission_pct
            if potential_cost + commission_on_buy > cash:
                # Adjust shares to fit within available cash
                shares_to_buy = (cash / (exec_price * (1 + commission_pct)))

            # Execute buy if shares_to_buy is significant
            if shares_to_buy > 0.0001: # Avoid tiny fractional shares
                cost        = shares_to_buy * exec_price
                commission  = cost * commission_pct
                cash        -= (cost + commission)

                # Set active trade details
                active_trade["entry_date"]  = date
                active_trade["entry_price"] = exec_price
                active_trade["shares"]      = shares_to_buy

                # Calculate Stop-Loss and Take-Profit levels
                if stop_loss_pct > 0:
                    active_trade["stop_loss"] = exec_price * (1 - stop_loss_pct)
                if take_profit_pct > 0:
                    active_trade["take_profit"] = exec_price * (1 + take_profit_pct)

                trade_log.append({
                    "date": date, "type": "BUY", "price": exec_price,
                    "shares": shares_to_buy, "commission": commission, "entry_date": date,
                    "holding_period": np.nan, # Included for consistency
                    "stop_loss": active_trade["stop_loss"],
                    "take_profit": active_trade["take_profit"]
                })

        elif signal == -1 and active_trade["shares"] > 0: # Sell signal and in a position
            exec_price  = price * (1 - slippage_pct)
            gross       = active_trade["shares"] * exec_price
            commission  = gross * commission_pct
            cash        += gross - commission
            pnl         = (exec_price - active_trade["entry_price"]) * active_trade["shares"] - commission
            holding_period = (date - active_trade["entry_date"]).days
            trade_log.append({
                "date": date, "type": "SELL", "price": exec_price,
                "shares": active_trade["shares"], "commission": commission,
                "pnl": pnl, "entry": active_trade["entry_price"], "holding_period": holding_period
            })
            active_trade = {"entry_date": None, "entry_price": None, "shares": 0,
                            "stop_loss": None, "take_profit": None}

        # Current portfolio value is cash plus value of shares held at current price.
        portfolio.append({"date": date, "value": cash + active_trade["shares"] * price})

    # If there's an open position at the end of the backtest, close it out
    if active_trade["shares"] > 0:
        last_date = df.index[-1]
        last_price = df["Close"].iloc[-1]
        exec_price = last_price * (1 - slippage_pct)
        gross       = active_trade["shares"] * exec_price
        commission  = gross * commission_pct
        cash        += gross - commission
        pnl         = (exec_price - active_trade["entry_price"]) * active_trade["shares"] - commission
        holding_period = (last_date - active_trade["entry_date"]).days
        trade_log.append({
            "date": last_date, "type": "SELL (EOD)", "price": exec_price,
            "shares": active_trade["shares"], "commission": commission,
            "pnl": pnl, "entry": active_trade["entry_price"], "holding_period": holding_period
        })

    portfolio_df       = pd.DataFrame(portfolio).set_index("date")
    portfolio_df.index = pd.to_datetime(portfolio_df.index)
    trades_df          = pd.DataFrame(trade_log) if trade_log else pd.DataFrame()
    if not trades_df.empty:
        trades_df      = trades_df.set_index("date")


    # 6. CALCULATE PERFORMANCE METRICS
    final_value  = portfolio_df["value"].iloc[-1]
    total_return = (final_value - initial_capital) / initial_capital * 100

    bh_shares = (initial_capital * (1 - commission_pct)) / float(df["Close"].iloc[0])
    bh_final   = bh_shares * float(df["Close"].iloc[-1])
    bh_return = (bh_final - initial_capital) / initial_capital * 100
    bh_series = bh_shares * df["Close"].reindex(portfolio_df.index)

    daily_ret  = portfolio_df["value"].pct_change().dropna()
    excess_ret = daily_ret - risk_free_rate / 252
    sharpe     = (excess_ret.mean() / excess_ret.std()) * np.sqrt(252)

    rolling_max = portfolio_df["value"].cummax()
    drawdown    = (portfolio_df["value"] - rolling_max) / rolling_max * 100
    max_dd      = drawdown.min()

    sell_trades = (trades_df[trades_df["type"].str.contains("SELL")]
                   if not trades_df.empty else pd.DataFrame())
    if not sell_trades.empty:
        winning  = (sell_trades["pnl"] > 0).sum()
        win_rate = winning / len(sell_trades) * 100
        n_trades = len(sell_trades)
    else:
        win_rate = 0.0
        n_trades = 0

    # Calculate advanced metrics
    advanced_metrics = calculate_advanced_metrics(portfolio_df, trades_df, initial_capital, risk_free_rate, n_trades)

    if log_output:
        print("=" * 44)
        print("  PERFORMANCE SUMMARY")
        print("=" * 44)
        print(f"  Initial Capital    : ${initial_capital:>12,.2f}")
        print(f"  Final Value        : ${final_value:>12,.2f}")
        print(f"  Strategy Return    : {total_return:>+11.2f} %")
        print(f"  CAGR               : {advanced_metrics['cagr']:>+11.2f} %")
        print(f"  Buy-and-Hold Ret.  : {bh_return:>+11.2f} %")
        print(f"  Volatility         : {advanced_metrics['volatility']:>11.2f} %")
        print(f"  Sharpe Ratio       : {sharpe:>12.3f}")
        print(f"  Sortino Ratio       : {advanced_metrics['sortino_ratio']:>12.3f}")
        print(f"  Max Drawdown       : {max_dd:>11.2f} %")
        print(f"  Calmar Ratio       : {advanced_metrics['calmar_ratio']:>12.3f}")
        print(f"  Win Rate           : {win_rate:>11.1f} %  ({n_trades} trades)")
        print(f"  Profit Factor      : {advanced_metrics['profit_factor']:>12.2f}")
        print(f"  Avg Trade Return   : ${advanced_metrics['avg_trade_return']:>11.2f}")
        print(f"  Expectancy         : ${advanced_metrics['expectancy']:>11.2f}")
        print(f"  Avg Holding Period : {advanced_metrics['avg_holding_period']:>11.1f} days")
        print("=" * 44)

        if not sell_trades.empty:
            print("\n  TRADE LOG (last 10 closed trades):")
            print(f"  {'Date':<12} {'Type':<11} {'Price':>8}  {'Shares':>8}  {'P&L':>10}  {'Hold':>5}")
            print("  " + "-" * 64)
            for date, t in trades_df.tail(10).iterrows():
                pnl_str = f"${t['pnl']:>+9.2f}" if "pnl" in t and pd.notna(t['pnl']) else " " * 11
                hold_str = f"{t['holding_period']:>5.0f}" if "holding_period" in t and pd.notna(t['holding_period']) else " " * 5
                print(f"  {str(date.date()):<12} {t['type']:<11} {t['price']:>8.2f}"
                      f"  {t['shares']:>8.4f}  {pnl_str}  {hold_str}")
        print()

    if plot_output:
        # Colour palette
        BG       = "#181c26"
        C_PRICE  = "#dce1e7"
        C_RSI    = "#9c6ade"   # purple RSI line
        C_BUY    = "#4caf50"   # green buy triangles
        C_SELL   = "#f44336"   # red sell triangles
        C_EQUITY = "#f6c90e"   # gold equity curve
        C_BH     = "#546e8a"   # muted blue benchmark
        C_DD     = "#f44336"   # red drawdown

        fig = plt.figure(figsize=(14, 11))
        fig.patch.set_facecolor("#0f1117")
        gs = gridspec.GridSpec(4, 1, figure=fig, height_ratios=[3, 1, 2, 1.2], hspace=0.08)

        AX_PRICE  = fig.add_subplot(gs[0])
        AX_RSI    = fig.add_subplot(gs[1], sharex=AX_PRICE)
        AX_EQUITY = fig.add_subplot(gs[2], sharex=AX_PRICE)
        AX_DD     = fig.add_subplot(gs[3], sharex=AX_PRICE)

        for ax in [AX_PRICE, AX_RSI, AX_EQUITY, AX_DD]:
            ax.set_facecolor(BG)
            ax.tick_params(colors="#aaaaaa", labelsize=9)
            ax.yaxis.label.set_color("#aaaaaa")
            for spine in ax.spines.values():
                spine.set_edgecolor("#333344")
            ax.grid(color="#262636", linewidth=0.5, linestyle="--")

        # Panel 1: Price + trade signals
        AX_PRICE.plot(df.index, df["Close"],    color=C_PRICE, lw=1.0, label=f"{ticker} Close")

        # Plot original signals for visualization (not for trade execution)
        buys_original  = df[df["position"].diff() ==  1]
        sells_original = df[df["position"].diff() == -1]
        AX_PRICE.scatter(buys_original.index,  buys_original["Close"],  marker="^", color=C_BUY,  s=80,
                         zorder=5, label="Buy (original signal)")
        AX_PRICE.scatter(sells_original.index, sells_original["Close"], marker="v", color=C_SELL, s=80,
                         zorder=5, label="Sell (original signal)")

        AX_PRICE.set_ylabel("Price (USD)", fontsize=10)
        title_full = (
            f"{ticker}  |  RSI({rsi_period}) Buy < {rsi_buy_threshold} / Sell > {rsi_sell_threshold}  "
            f"[{start_date} to {end_date}]{plot_title_suffix}"
        )
        AX_PRICE.set_title(
            title_full,
            color="white", fontsize=12, pad=10, fontweight="bold"
        )
        AX_PRICE.legend(loc="upper left", fontsize=8, framealpha=0.25,
                        facecolor="#0f1117", labelcolor="white")

        # Panel 2: RSI
        AX_RSI.plot(df.index, df['RSI'], color=C_RSI, lw=1.5, label=f'RSI ({rsi_period})')
        AX_RSI.axhline(rsi_buy_threshold, color=C_BUY, linestyle='--', lw=1, alpha=0.7, label=f'Buy Threshold ({rsi_buy_threshold})')
        AX_RSI.axhline(rsi_sell_threshold, color=C_SELL, linestyle='--', lw=1, alpha=0.7, label=f'Sell Threshold ({rsi_sell_threshold})')
        AX_RSI.set_ylabel("RSI", fontsize=10)
        AX_RSI.legend(loc="upper left", fontsize=8, framealpha=0.25,
                      facecolor="#0f1117", labelcolor="white")

        # Panel 3: Equity curve vs. buy-and-hold benchmark
        AX_EQUITY.plot(portfolio_df.index, portfolio_df["value"],
                       color=C_EQUITY, lw=1.6, label=f"Strategy   {total_return:+.1f}%")
        AX_EQUITY.plot(bh_series.index, bh_series.values,
                       color=C_BH, lw=1.4, linestyle="--",
                       label=f"Buy & Hold  {bh_return:+.1f}%")
        AX_EQUITY.axhline(initial_capital, color="#555566", lw=0.8, linestyle=":")
        AX_EQUITY.set_ylabel("Portfolio Value ($", fontsize=10)
        AX_EQUITY.legend(loc="upper left", fontsize=8, framealpha=0.25,
                         facecolor="#0f1117", labelcolor="white")

        stats_label = (f"Sharpe: {sharpe:.2f}   |   Max Drawdown: {max_dd:.1f}%   |   Win Rate: {win_rate:.0f}%  "
                       f"({n_trades} trades)")
        AX_EQUITY.text(0.99, 0.04, stats_label, transform=AX_EQUITY.transAxes, fontsize=8, color="#bbbbbb", ha="right", bbox=dict(boxstyle="round,pad=0.35", fc="#0f1117", alpha=0.55))

        # Panel 4: Drawdown waterfall
        AX_DD.fill_between(portfolio_df.index, drawdown, 0, color=C_DD, alpha=0.35)
        AX_DD.plot(portfolio_df.index, drawdown, color=C_DD, lw=0.9)
        AX_DD.axhline(max_dd, color=C_DD, lw=0.8, linestyle="--", alpha=0.7, label=f"Max DD: {max_dd:.1f}%")
        AX_DD.set_ylabel("Drawdown (%)", fontsize=10)
        AX_DD.set_xlabel("Date", fontsize=10, color="#aaaaaa")
        AX_DD.legend(loc="lower left", fontsize=8, framealpha=0.25, facecolor="#0f1117", labelcolor="white")

        plt.setp(AX_PRICE.get_xticklabels(),  visible=False)
        plt.setp(AX_RSI.get_xticklabels(), visible=False)
        plt.setp(AX_EQUITY.get_xticklabels(), visible=False)

        output_path = "backtest_rsi_results.png"
        plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
        plt.show()
        if log_output:
            print(f"Chart saved -> {output_path}\n")

    return {
        "total_return": total_return,
        "bh_return": bh_return,
        "sharpe_ratio": sharpe,
        "max_drawdown": max_dd,
        "win_rate": win_rate,
        "n_trades": n_trades,
        "portfolio_df": portfolio_df,
        "bh_series": bh_series,
        **advanced_metrics # Include all advanced metrics
    }

### Changes to `run_sma_backtest` for Lookahead Bias Elimination

To ensure the backtester makes realistic trading decisions, the following modifications were implemented in the `run_sma_backtest` function:

1.  **Shifted Position Calculation**: A new column, `df['position_for_trade_decision']`, was introduced. This column stores the trading position (`1` for long, `0` for neutral/short) determined by the SMA crossover rule *from the previous day's closing prices*. This means that today's trading decision is based on yesterday's market close.

2.  **Delayed Trade Signal Generation**: The `df['trade_signal']` is now derived from the `position_for_trade_decision` column. A `trade_signal` of `1` (buy) or `-1` (sell) is generated on day `t` if the position determined on day `t-1` changed from the position determined on day `t-2`. This ensures that a signal generated on day `t` is acted upon using the closing price of day `t` itself, reflecting a more realistic execution time for decisions made at the close of the prior day.

3.  **Modified Trade Execution Logic**: The trade simulation loop now explicitly uses `row["trade_signal"]` for making buy/sell decisions. This guarantees that trades are executed only after the signal has been generated based on past information, removing any lookahead bias that might have existed from generating and executing on the same bar.

4.  **Updated Plotting for Signals**: The trade signals shown on the price chart (`AX_PRICE.scatter`) now reflect the *original* signals (when `SMA_fast > SMA_slow`) for visualization purposes. This helps to show when the crossover originally occurred, even if the trade execution itself is delayed by one bar. The actual trade entries and exits in the `trade_log` and performance metrics are based on the lookahead-free `trade_signal`.

These changes ensure that the backtest accurately reflects performance under conditions where trading decisions can only be based on information that would have been available at the time of the decision.

In [ ]:
%%capture
pip install yfinance

In [ ]:
# Run the backtest with default parameters
results = run_sma_backtest(
    ticker=TICKER,
    start_date=START_DATE,
    end_date=END_DATE,
    fast_sma=FAST_SMA,
    slow_sma=SLOW_SMA,
    initial_capital=INITIAL_CAPITAL,
    commission_pct=COMMISSION_PCT,
    risk_free_rate=RISK_FREE_RATE,
    slippage_pct=SLIPPAGE_PCT,
    log_output=True,
    plot_output=True
)

TypeError: run_sma_backtest() missing 7 required positional arguments: 'position_size_pct', 'max_exposure_pct', 'volatility_adjusted_sizing', 'atr_period', 'atr_multiplier', 'stop_loss_pct', and 'take_profit_pct'

In [ ]:
# 8. EXECUTE BACKTESTER FOR EACH STRATEGY

all_results = {}

for strategy in strategies:
    print(f"\nRunning backtest for: {strategy['name']}")

    if strategy['type'] == 'SMA':
        strat_results = run_sma_backtest(
            ticker=TICKER,
            start_date=START_DATE,
            end_date=END_DATE,
            fast_sma=strategy['fast_sma'],
            slow_sma=strategy['slow_sma'],
            initial_capital=INITIAL_CAPITAL,
            commission_pct=COMMISSION_PCT,
            risk_free_rate=RISK_FREE_RATE,
            slippage_pct=SLIPPAGE_PCT,
            log_output=True,
            plot_output=False, # We'll plot all together later
            plot_title_suffix=f" ({strategy['name']})"
        )
    elif strategy['type'] == 'RSI':
        strat_results = run_rsi_backtest(
            ticker=TICKER,
            start_date=START_DATE,
            end_date=END_DATE,
            rsi_period=strategy['rsi_period'],
            rsi_buy_threshold=strategy['rsi_buy_threshold'],
            rsi_sell_threshold=strategy['rsi_sell_threshold'],
            initial_capital=INITIAL_CAPITAL,
            commission_pct=COMMISSION_PCT,
            risk_free_rate=RISK_FREE_RATE,
            slippage_pct=SLIPPAGE_PCT,
            log_output=True,
            plot_output=False, # We'll plot all together later
            plot_title_suffix=f" ({strategy['name']})"
        )
    else:
        print(f"  Unknown strategy type: {strategy['type']} for {strategy['name']}. Skipping.")
        continue

    all_results[strategy['name']] = strat_results

print("\nAll backtests completed.")

In [ ]:
metrics_data = []
for name, res in all_results.items():
    metrics_data.append({
        'Strategy': name,
        'Return (%)': f"{res['total_return']:.2f}",
        'CAGR (%)': f"{res['cagr']:.2f}",
        'Volatility (%)': f"{res['volatility']:.2f}",
        'Sharpe Ratio': f"{res['sharpe_ratio']:.3f}",
        'Sortino Ratio': f"{res['sortino_ratio']:.3f}",
        'Max Drawdown (%)': f"{res['max_drawdown']:.2f}",
        'Calmar Ratio': f"{res['calmar_ratio']:.3f}",
        'Win Rate (%)': f"{res['win_rate']:.1f}",
        'Num Trades': res['n_trades'],
        'Profit Factor': f"{res['profit_factor']:.2f}",
        'Avg Trade Return ($)': f"{res['avg_trade_return']:.2f}",
        'Expectancy ($)': f"{res['expectancy']:.2f}",
        'Avg Holding Period (Days)': f"{res['avg_holding_period']:.1f}"
    })

metrics_df = pd.DataFrame(metrics_data)
metrics_df['Sharpe Ratio'] = pd.to_numeric(metrics_df['Sharpe Ratio'])
metrics_df.set_index('Strategy', inplace=True)
metrics_df.sort_values(by='Sharpe Ratio', ascending=False, inplace=True)

print("\n" + "="*40)
print("  MULTI-STRATEGY PERFORMANCE DASHBOARD")
print("="*40)
display(metrics_df)
print("\n")

In [ ]:
# 10. GENERATE COMPARATIVE EQUITY CURVE PLOTS

fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor("#0f1117")
ax = fig.add_subplot(111)

# Color palette for equity curves
colors = ['#9c6ade', '#f6c90e', '#3fa7d6', '#4caf50', '#f44336', '#ff5722']

ax.set_facecolor("#181c26")
ax.tick_params(colors="#aaaaaa", labelsize=10)
ax.yaxis.label.set_color("#aaaaaa")
ax.xaxis.label.set_color("#aaaaaa")
for spine in ax.spines.values():
    spine.set_edgecolor("#333344")
ax.grid(color="#262636", linewidth=0.5, linestyle="--")

# Plot buy-and-hold once (using the first strategy's bh_series for consistency)
# Sort all_results by Sharpe Ratio to ensure consistent plotting order with the dashboard
sorted_all_results = sorted(all_results.items(), key=lambda item: item[1]['sharpe_ratio'], reverse=True)

first_strategy_name = sorted_all_results[0][0]
bh_series = all_results[first_strategy_name]['bh_series']
ax.plot(bh_series.index, bh_series.values, color='#546e8a', lw=1.5, linestyle="--",
        label=f"Buy & Hold ({all_results[first_strategy_name]['bh_return']:+.1f}%) -- Return Only") # Changed label to reflect it's the benchmark

# Plot each strategy's equity curve, ensuring consistent order
for i, (name, res) in enumerate(sorted_all_results):
    portfolio_df = res['portfolio_df']
    color = colors[i % len(colors)]
    ax.plot(portfolio_df.index, portfolio_df['value'], color=color, lw=2.0,
            label=f"{name} ({res['total_return']:+.1f}%)")

ax.axhline(INITIAL_CAPITAL, color="#555566", lw=0.8, linestyle=":")
ax.set_title("Multi-Strategy Equity Curve Comparison", color="white", fontsize=16, pad=15, fontweight="bold")
ax.set_ylabel("Portfolio Value ($")
ax.set_xlabel("Date")
ax.legend(loc="upper left", fontsize=9, framealpha=0.25, facecolor="#0f1117", labelcolor="white")

output_path = "multi_strategy_equity_curves.png"
plt.savefig(output_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"\nComparative equity curves chart saved -> {output_path}\n")

## 11. Parameter Optimization Engine

To find the optimal `FAST_SMA` and `SLOW_SMA` parameters, we'll implement a simple grid search optimization. This involves:

1.  Defining the ranges for `FAST_SMA` and `SLOW_SMA`.
2.  Generating all valid combinations (ensuring `FAST_SMA < SLOW_SMA`).
3.  Running `run_sma_backtest` for each combination.
4.  Storing and analyzing the results.
5.  Ranking the parameter sets by Sharpe Ratio.


In [ ]:
# Define parameter ranges
fast_sma_range = range(10, 101, 10) # From 10 to 100, step 10
slow_sma_range = range(50, 301, 25) # From 50 to 300, step 25

# Store all valid combinations
sma_combinations = []

for fast_sma in fast_sma_range:
    for slow_sma in slow_sma_range:
        if fast_sma < slow_sma: # Ensure fast SMA is always less than slow SMA
            sma_combinations.append({
                'name': f'SMA {fast_sma}/{slow_sma}',
                'type': 'SMA',
                'fast_sma': fast_sma,
                'slow_sma': slow_sma
            })

print(f"Generated {len(sma_combinations)} valid SMA combinations for optimization.")
# print("Example combinations:", sma_combinations[:5]) # Display first 5 for verification

### Running Backtests for All Combinations

Now, we'll iterate through all generated SMA combinations and run the `run_sma_backtest` function for each. We'll collect the key performance metrics (Sharpe Ratio, Total Return, Max Drawdown) for each combination.

This process can take some time depending on the number of combinations and the backtest period. Progress will be printed to the console.

In [ ]:
optimization_results = []

print("\nStarting parameter optimization backtests...")

for i, combo in enumerate(sma_combinations):
    if (i + 1) % 20 == 0: # Print progress every 20 combinations
        print(f"  Processing combination {i + 1}/{len(sma_combinations)}: {combo['name']}")

    # Run the backtest for the current SMA combination
    # Set log_output=False to avoid excessive output during optimization
    current_results = run_sma_backtest(
        ticker=TICKER,
        start_date=START_DATE,
        end_date=END_DATE,
        fast_sma=combo['fast_sma'],
        slow_sma=combo['slow_sma'],
        initial_capital=INITIAL_CAPITAL,
        commission_pct=COMMISSION_PCT,
        risk_free_rate=RISK_FREE_RATE,
        slippage_pct=SLIPPAGE_PCT,
        log_output=False, # Suppress individual backtest logs for brevity
        plot_output=False
    )

    # Store relevant metrics along with the parameters
    optimization_results.append({
        'Fast_SMA': combo['fast_sma'],
        'Slow_SMA': combo['slow_sma'],
        'Strategy_Name': combo['name'],
        'Total_Return (%)': current_results['total_return'],
        'CAGR (%)': current_results['cagr'],
        'Volatility (%)': current_results['volatility'],
        'Sharpe_Ratio': current_results['sharpe_ratio'],
        'Sortino Ratio': current_results['sortino_ratio'],
        'Max_Drawdown (%)': current_results['max_drawdown'],
        'Calmar Ratio': current_results['calmar_ratio'],
        'Win_Rate (%)': current_results['win_rate'],
        'Num_Trades': current_results['n_trades'],
        'Profit Factor': current_results['profit_factor'],
        'Avg Trade Return ($)': current_results['avg_trade_return'],
        'Expectancy ($)': current_results['expectancy'],
        'Avg Holding Period (Days)': current_results['avg_holding_period']
    })

print("\nAll optimization backtests completed.")

### Analyze and Rank Results

Now we'll convert the optimization results into a Pandas DataFrame, sort them by `Sharpe_Ratio` in descending order, and display the top 10 performing parameter combinations.

In [ ]:
# Convert results to DataFrame
optimization_df = pd.DataFrame(optimization_results)

# Sort by Sharpe Ratio in descending order
optimization_df_sorted = optimization_df.sort_values(by='Sharpe_Ratio', ascending=False)

# Round float columns for cleaner display
for col in ['Total_Return (%)', 'CAGR (%)', 'Volatility (%)', 'Sharpe_Ratio', 'Sortino Ratio',
            'Max_Drawdown (%)', 'Calmar Ratio', 'Win_Rate (%)', 'Profit Factor',
            'Avg Trade Return ($)', 'Expectancy ($)', 'Avg Holding Period (Days)']:
    if col in optimization_df_sorted.columns:
        optimization_df_sorted[col] = optimization_df_sorted[col].round(2)

print("\n" + "="*50)
print("  TOP 10 SMA CROSSOVER STRATEGIES BY SHARPE RATIO")
print("="*50)
display(optimization_df_sorted.head(10))
print("\n")

## 12. Multi-Asset, Multi-Strategy Backtesting

Now, we'll extend our backtesting framework to run all defined strategies across a list of specified assets (e.g., `SPY`, `QQQ`, `IWM`, `GLD`, `TLT`). This will allow us to compare strategy performance and robustness across different market segments.

The process will involve:
1.  Iterating through each `ticker` in the `TICKERS` list.
2.  For each `ticker`, iterating through the `strategies` list.
3.  Executing the appropriate backtest function (`run_sma_backtest` or `run_rsi_backtest`).
4.  Collecting all results into a single list.
5.  Aggregating the results into a Pandas DataFrame for comparison.

In [ ]:
multi_asset_results = []

print("\nStarting multi-asset, multi-strategy backtests...")

for ticker in TICKERS:
    print(f"\n{'='*50}")
    print(f"  Backtesting for Asset: {ticker}")
    print(f"{'='*50}")

    for strategy in strategies:
        print(f"  Running strategy: {strategy['name']} on {ticker}")

        strat_results = None
        if strategy['type'] == 'SMA':
            strat_results = run_sma_backtest(
                ticker=ticker,
                start_date=START_DATE,
                end_date=END_DATE,
                fast_sma=strategy['fast_sma'],
                slow_sma=strategy['slow_sma'],
                initial_capital=INITIAL_CAPITAL,
                commission_pct=COMMISSION_PCT,
                risk_free_rate=RISK_FREE_RATE,
                slippage_pct=SLIPPAGE_PCT,
                log_output=False, # Suppress individual backtest logs for brevity
                plot_output=False
            )
        elif strategy['type'] == 'RSI':
            strat_results = run_rsi_backtest(
                ticker=ticker,
                start_date=START_DATE,
                end_date=END_DATE,
                rsi_period=strategy['rsi_period'],
                rsi_buy_threshold=strategy['rsi_buy_threshold'],
                rsi_sell_threshold=strategy['rsi_sell_threshold'],
                initial_capital=INITIAL_CAPITAL,
                commission_pct=COMMISSION_PCT,
                risk_free_rate=RISK_FREE_RATE,
                slippage_pct=SLIPPAGE_PCT,
                log_output=False, # Suppress individual backtest logs for brevity
                plot_output=False
            )
        else:
            print(f"    Unknown strategy type: {strategy['type']} for {strategy['name']}. Skipping.")
            continue

        if strat_results:
            multi_asset_results.append({
                'Asset': ticker,
                'Strategy': strategy['name'],
                'Return (%)': f"{strat_results['total_return']:.2f}",
                'CAGR (%)': f"{strat_results['cagr']:.2f}",
                'Volatility (%)': f"{strat_results['volatility']:.2f}",
                'Sharpe Ratio': f"{strat_results['sharpe_ratio']:.3f}",
                'Sortino Ratio': f"{strat_results['sortino_ratio']:.3f}",
                'Max Drawdown (%)': f"{strat_results['max_drawdown']:.2f}",
                'Calmar Ratio': f"{strat_results['calmar_ratio']:.3f}",
                'Win Rate (%)': f"{strat_results['win_rate']:.1f}",
                'Num Trades': strat_results['n_trades'],
                'Profit Factor': f"{strat_results['profit_factor']:.2f}",
                'Avg Trade Return ($)': f"{strat_results['avg_trade_return']:.2f}",
                'Expectancy ($)': f"{strat_results['expectancy']:.2f}",
                'Avg Holding Period (Days)': f"{strat_results['avg_holding_period']:.1f}"
            })

print("\nAll multi-asset, multi-strategy backtests completed.")

### Multi-Asset Performance Dashboard

Below is the aggregated performance dashboard showing how each strategy performed on every asset. The results are sorted by Sharpe Ratio in descending order.

## 13. Walk-Forward Optimization

Walk-forward optimization is a rigorous method for evaluating trading strategies that simulates how a strategy would be developed and applied in real-time. It addresses the issue of *data snooping bias* that can arise from optimizing parameters on the entire historical dataset.

**The process involves:**
1.  **Defining Rolling Windows**: The entire historical dataset is divided into a series of overlapping or non-overlapping time windows.
2.  **In-Sample Optimization (Training)**: For each window, parameters are optimized using only the 'in-sample' (training) data within that window.
3.  **Out-of-Sample Validation (Testing)**: The best parameters found in the in-sample optimization are then applied to the subsequent 'out-of-sample' (validation) period, which the optimization process has not seen.
4.  **Performance Comparison**: The performance of the strategy with the optimized parameters on the out-of-sample data is recorded and compared against the in-sample performance.
5.  **Iteration**: This process is repeated for subsequent windows, moving forward in time.

This approach provides a more realistic estimate of how a strategy might perform in the future, as it continuously re-optimizes parameters based on the most recent available data, just as a trader would. The goal is to identify strategies that are robust and perform well on unseen data, rather than being overfitted to historical data.

In [ ]:
from datetime import timedelta

def run_walk_forward_optimization(
    ticker, strategy_type, start_date, end_date,
    in_sample_period_years, out_of_sample_period_years, step_years,
    optimization_params_range,
    initial_capital, commission_pct, risk_free_rate, slippage_pct,
    position_size_pct, max_exposure_pct, volatility_adjusted_sizing,
    atr_period, atr_multiplier, stop_loss_pct, take_profit_pct
):
    """
    Performs walk-forward optimization for a given strategy type.

    Args:
        ticker (str): The ticker symbol for the asset.
        strategy_type (str): 'SMA' or 'RSI'.
        start_date (str): Global start date for the entire walk-forward test.
        end_date (str): Global end date for the entire walk-forward test.
        in_sample_period_years (int): Duration of the in-sample optimization period in years.
        out_of_sample_period_years (int): Duration of the out-of-sample validation period in years.
        step_years (int): How many years to advance the window for the next iteration.
        optimization_params_range (dict): Dictionary defining the ranges for strategy-specific parameters.
                                          e.g., {'fast_sma': range(10, 101, 10), 'slow_sma': range(50, 301, 25)}
                                          or {'rsi_period': range(7, 22, 1), 'rsi_buy_threshold': range(20, 40, 5), 'rsi_sell_threshold': range(60, 80, 5)}
        Other args: Same as run_sma_backtest/run_rsi_backtest for consistency.

    Returns:
        pd.DataFrame: A DataFrame containing the walk-forward results (in-sample and out-of-sample).
    """
    all_walk_forward_results = []

    current_start_date = pd.to_datetime(start_date)
    global_end_date = pd.to_datetime(end_date)

    print(f"\nStarting Walk-Forward Optimization for {ticker} with {strategy_type} strategy...")
    window_num = 1

    while True:
        in_sample_start = current_start_date
        in_sample_end = in_sample_start + pd.DateOffset(years=in_sample_period_years) - pd.DateOffset(days=1) # -1 day to avoid overlap

        out_of_sample_start = in_sample_end + pd.DateOffset(days=1)
        out_of_sample_end = out_of_sample_start + pd.DateOffset(years=out_of_sample_period_years) - pd.DateOffset(days=1)

        # Break condition if out-of-sample period goes beyond global end date
        if out_of_sample_end > global_end_date:
            out_of_sample_end = global_end_date # Adjust last OOS window to fit
            if out_of_sample_start >= out_of_sample_end: # Ensure at least one day for OOS
                break

        print(f"\n{'='*70}")
        print(f"  WALK-FORWARD WINDOW {window_num}")
        print(f"  In-Sample: {in_sample_start.strftime('%Y-%m-%d')} to {in_sample_end.strftime('%Y-%m-%d')}")
        print(f"  Out-of-Sample: {out_of_sample_start.strftime('%Y-%m-%d')} to {out_of_sample_end.strftime('%Y-%m-%d')}")
        print(f"{'='*70}")

        # --- 1. In-Sample Optimization (Training) ---
        print("  Performing in-sample optimization...")
        best_in_sample_params = None
        best_in_sample_sharpe = -np.inf
        in_sample_optimization_results = []

        # Generate combinations based on strategy type
        if strategy_type == 'SMA':
            param_combinations = []
            for fast_sma in optimization_params_range['fast_sma']:
                for slow_sma in optimization_params_range['slow_sma']:
                    if fast_sma < slow_sma:
                        param_combinations.append({'fast_sma': fast_sma, 'slow_sma': slow_sma})
        elif strategy_type == 'RSI':
            param_combinations = []
            for rsi_period in optimization_params_range['rsi_period']:
                for rsi_buy_threshold in optimization_params_range['rsi_buy_threshold']:
                    for rsi_sell_threshold in optimization_params_range['rsi_sell_threshold']:
                        if rsi_buy_threshold < rsi_sell_threshold: # Ensure thresholds are logical
                            param_combinations.append({
                                'rsi_period': rsi_period,
                                'rsi_buy_threshold': rsi_buy_threshold,
                                'rsi_sell_threshold': rsi_sell_threshold
                            })
        else:
            raise ValueError("Unsupported strategy type.")

        for combo in param_combinations:
            if strategy_type == 'SMA':
                results = run_sma_backtest(
                    ticker=ticker, start_date=in_sample_start.strftime('%Y-%m-%d'), end_date=in_sample_end.strftime('%Y-%m-%d'),
                    fast_sma=combo['fast_sma'], slow_sma=combo['slow_sma'],
                    initial_capital=initial_capital, commission_pct=commission_pct, risk_free_rate=risk_free_rate, slippage_pct=slippage_pct,
                    position_size_pct=position_size_pct, max_exposure_pct=max_exposure_pct,
                    volatility_adjusted_sizing=volatility_adjusted_sizing, atr_period=atr_period, atr_multiplier=atr_multiplier,
                    stop_loss_pct=stop_loss_pct, take_profit_pct=take_profit_pct,
                    log_output=False, plot_output=False
                )
            elif strategy_type == 'RSI':
                results = run_rsi_backtest(
                    ticker=ticker, start_date=in_sample_start.strftime('%Y-%m-%d'), end_date=in_sample_end.strftime('%Y-%m-%d'),
                    rsi_period=combo['rsi_period'], rsi_buy_threshold=combo['rsi_buy_threshold'], rsi_sell_threshold=combo['rsi_sell_threshold'],
                    initial_capital=initial_capital, commission_pct=commission_pct, risk_free_rate=risk_free_rate, slippage_pct=slippage_pct,
                    position_size_pct=position_size_pct, max_exposure_pct=max_exposure_pct,
                    volatility_adjusted_sizing=volatility_adjusted_sizing, atr_period=atr_period, atr_multiplier=atr_multiplier,
                    stop_loss_pct=stop_loss_pct, take_profit_pct=take_profit_pct,
                    log_output=False, plot_output=False
                )

            if results['sharpe_ratio'] > best_in_sample_sharpe:
                best_in_sample_sharpe = results['sharpe_ratio']
                best_in_sample_params = combo

            # Store in-sample optimization results
            in_sample_optimization_results.append({
                'Window': window_num,
                'Period_Type': 'In-Sample',
                'Start_Date': in_sample_start.strftime('%Y-%m-%d'),
                'End_Date': in_sample_end.strftime('%Y-%m-%d'),
                **combo, # Include parameters
                'Sharpe_Ratio': results['sharpe_ratio'],
                'Total_Return': results['total_return'],
                'Max_Drawdown': results['max_drawdown']
            })

        if best_in_sample_params is None:
            print("    No valid parameters found for in-sample optimization. Skipping window.")
            current_start_date += pd.DateOffset(years=step_years)
            window_num += 1
            continue

        print(f"  Best In-Sample Params: {best_in_sample_params} (Sharpe: {best_in_sample_sharpe:.3f})")

        # --- 2. Out-of-Sample Validation (Testing) ---
        print("  Running out-of-sample validation...")
        if strategy_type == 'SMA':
            oos_results = run_sma_backtest(
                ticker=ticker, start_date=out_of_sample_start.strftime('%Y-%m-%d'), end_date=out_of_sample_end.strftime('%Y-%m-%d'),
                fast_sma=best_in_sample_params['fast_sma'], slow_sma=best_in_sample_params['slow_sma'],
                initial_capital=initial_capital, commission_pct=commission_pct, risk_free_rate=risk_free_rate, slippage_pct=slippage_pct,
                position_size_pct=position_size_pct, max_exposure_pct=max_exposure_pct,
                volatility_adjusted_sizing=volatility_adjusted_sizing, atr_period=atr_period, atr_multiplier=atr_multiplier,
                stop_loss_pct=stop_loss_pct, take_profit_pct=take_profit_pct,
                log_output=False, plot_output=False
            )
        elif strategy_type == 'RSI':
            oos_results = run_rsi_backtest(
                ticker=ticker, start_date=out_of_sample_start.strftime('%Y-%m-%d'), end_date=out_of_sample_end.strftime('%Y-%m-%d'),
                rsi_period=best_in_sample_params['rsi_period'], rsi_buy_threshold=best_in_sample_params['rsi_buy_threshold'], rsi_sell_threshold=best_in_sample_params['rsi_sell_threshold'],
                initial_capital=initial_capital, commission_pct=commission_pct, risk_free_rate=risk_free_rate, slippage_pct=slippage_pct,
                position_size_pct=position_size_pct, max_exposure_pct=max_exposure_pct,
                volatility_adjusted_sizing=volatility_adjusted_sizing, atr_period=atr_period, atr_multiplier=atr_multiplier,
                stop_loss_pct=stop_loss_pct, take_profit_pct=take_profit_pct,
                log_output=False, plot_output=False
            )

        print(f"  Out-of-Sample Sharpe: {oos_results['sharpe_ratio']:.3f}")

        # Store results for this window
        result_entry = {
            'Window': window_num,
            'Strategy_Type': strategy_type,
            'Ticker': ticker,
            'In_Sample_Start': in_sample_start.strftime('%Y-%m-%d'),
            'In_Sample_End': in_sample_end.strftime('%Y-%m-%d'),
            'OOS_Start': out_of_sample_start.strftime('%Y-%m-%d'),
            'OOS_End': out_of_sample_end.strftime('%Y-%m-%d'),
            'Best_IS_Params': str(best_in_sample_params), # Store as string for easy display
            'IS_Sharpe': best_in_sample_sharpe,
            'OOS_Sharpe': oos_results['sharpe_ratio'],
            'OOS_Return': oos_results['total_return'],
            'OOS_Max_Drawdown': oos_results['max_drawdown']
        }
        all_walk_forward_results.append(result_entry)

        current_start_date += pd.DateOffset(years=step_years)
        window_num += 1

    print("\nWalk-Forward Optimization Completed.")
    return pd.DataFrame(all_walk_forward_results)

### 13.1 Example Walk-Forward Optimization for SMA Strategy

Let's run a walk-forward optimization for the SMA crossover strategy using the newly defined function. We'll define parameter ranges for the `fast_sma` and `slow_sma` for optimization during the in-sample period.


In [ ]:
# Define walk-forward parameters
WF_TICKER = 'SPY'
WF_START_DATE = '2005-01-01'
WF_END_DATE = '2024-12-31'
WF_IN_SAMPLE_YEARS = 5
WF_OUT_OF_SAMPLE_YEARS = 1
WF_STEP_YEARS = 1 # Move the window forward by 1 year each time

# Define optimization ranges for SMA strategy
WF_SMA_OPTIMIZATION_PARAMS = {
    'fast_sma': range(20, 61, 10), # e.g., 20, 30, 40, 50, 60
    'slow_sma': range(100, 201, 25) # e.g., 100, 125, 150, 175, 200
}

print(f"Running walk-forward optimization for SMA on {WF_TICKER}...")
sma_walk_forward_results = run_walk_forward_optimization(
    ticker=WF_TICKER,
    strategy_type='SMA',
    start_date=WF_START_DATE,
    end_date=WF_END_DATE,
    in_sample_period_years=WF_IN_SAMPLE_YEARS,
    out_of_sample_period_years=WF_OUT_OF_SAMPLE_YEARS,
    step_years=WF_STEP_YEARS,
    optimization_params_range=WF_SMA_OPTIMIZATION_PARAMS,
    initial_capital=INITIAL_CAPITAL,
    commission_pct=COMMISSION_PCT,
    risk_free_rate=RISK_FREE_RATE,
    slippage_pct=SLIPPAGE_PCT,
    position_size_pct=POSITION_SIZE_PCT,
    max_exposure_pct=MAX_EXPOSURE_PCT,
    volatility_adjusted_sizing=VOLATILITY_ADJUSTED_SIZING,
    atr_period=ATR_PERIOD,
    atr_multiplier=ATR_MULTIPLIER,
    stop_loss_pct=STOP_LOSS_PCT,
    take_profit_pct=TAKE_PROFIT_PCT
)

print("\nSMA Walk-Forward Results:")
display(sma_walk_forward_results.head())

### 13.2 Example Walk-Forward Optimization for RSI Strategy

Next, let's perform a walk-forward optimization for the RSI strategy. We'll define suitable ranges for `rsi_period`, `rsi_buy_threshold`, and `rsi_sell_threshold`.

In [ ]:
# Define optimization ranges for RSI strategy
WF_RSI_OPTIMIZATION_PARAMS = {
    'rsi_period': range(10, 21, 2), # e.g., 10, 12, ..., 20
    'rsi_buy_threshold': range(25, 36, 5), # e.g., 25, 30, 35
    'rsi_sell_threshold': range(65, 76, 5) # e.g., 65, 70, 75
}

print(f"Running walk-forward optimization for RSI on {WF_TICKER}...")
rsi_walk_forward_results = run_walk_forward_optimization(
    ticker=WF_TICKER,
    strategy_type='RSI',
    start_date=WF_START_DATE,
    end_date=WF_END_DATE,
    in_sample_period_years=WF_IN_SAMPLE_YEARS,
    out_of_sample_period_years=WF_OUT_OF_SAMPLE_YEARS,
    step_years=WF_STEP_YEARS,
    optimization_params_range=WF_RSI_OPTIMIZATION_PARAMS,
    initial_capital=INITIAL_CAPITAL,
    commission_pct=COMMISSION_PCT,
    risk_free_rate=RISK_FREE_RATE,
    slippage_pct=SLIPPAGE_PCT,
    position_size_pct=POSITION_SIZE_PCT,
    max_exposure_pct=MAX_EXPOSURE_PCT,
    volatility_adjusted_sizing=VOLATILITY_ADJUSTED_SIZING,
    atr_period=ATR_PERIOD,
    atr_multiplier=ATR_MULTIPLIER,
    stop_loss_pct=STOP_LOSS_PCT,
    take_profit_pct=TAKE_PROFIT_PCT
)

print("\nRSI Walk-Forward Results:")
display(rsi_walk_forward_results.head())

In [ ]:
multi_asset_df = pd.DataFrame(multi_asset_results)
multi_asset_df['Sharpe Ratio'] = pd.to_numeric(multi_asset_df['Sharpe Ratio'])
multi_asset_df.sort_values(by='Sharpe Ratio', ascending=False, inplace=True)

print("\n" + "="*60)
print("  MULTI-ASSET, MULTI-STRATEGY PERFORMANCE DASHBOARD")
print("="*60)
display(multi_asset_df)
print("\n")

In [ ]:
# 7. DEFINE MULTIPLE STRATEGIES
# We'll create a list of different SMA crossover strategies to compare.

strategies = [
    # Example 1: Original strategy (Fast SMA 50, Slow SMA 200)
    {
        'name': 'SMA 50/200',
        'type': 'SMA',
        'fast_sma': 50,
        'slow_sma': 200
    },
    # Example 2: Faster crossover (Fast SMA 20, Slow SMA 50)
    {
        'name': 'SMA 20/50',
        'type': 'SMA',
        'fast_sma': 20,
        'slow_sma': 50
    },
    # Example 3: Slower crossover (Fast SMA 100, Slow SMA 300)
    {
        'name': 'SMA 100/300',
        'type': 'SMA',
        'fast_sma': 100,
        'slow_sma': 300
    },
    # Example 4: Custom crossover (Fast SMA 40, Slow SMA 150)
    {
        'name': 'SMA 40/150',
        'type': 'SMA',
        'fast_sma': 40,
        'slow_sma': 150
    },
    # New: RSI Mean Reversion Strategy
    {
        'name': 'RSI 14 Mean Reversion',
        'type': 'RSI',
        'rsi_period': 14,
        'rsi_buy_threshold': 30,
        'rsi_sell_threshold': 70
    }
]